# 2 — Data Preparation
**Course:** Household Economics  
**Author:** Koray Aktas,Giacomo Faccin, Ca' Foscari Venezia  
**Data:** SHIW 1990–2022 — Bank of Italy

---
This notebook transforms the merged dataset (`data_SHIW2022`) into an analysis-ready file.

**Steps:**
1. Rename Italian variables to English & add labels
2. Deflate all monetary values to **2020 prices** using the `rival` coefficient
3. Construct additional variables: dummies, cohorts, age groups, children counts
4. Save as `dataready_SHIW2022`

## 1. Setup & Load Data

In [2]:
import os
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# ── Set paths ─────────────────────────────────────────────────────────────────
BASE_DIR = "C:/Users/Utente/OneDrive/Immagini/Documenti/Economic_Risk_work"   # ← change this
DataOut  = os.path.join(BASE_DIR, "DataOut")

df = pd.read_pickle(os.path.join(DataOut, "data_SHIW2022.pkl"))
print(f"Loaded: {df.shape[0]:,} rows × {df.shape[1]} cols")
df.head(3)

Loaded: 311,903 rows × 48 cols


,nquest,nord,anno,cfdic,sesso,nordp,par,nperc,nascarea,nascreg,...,pesopop,partime,mesilav,contratt,dimaz,ylm,oretot,orestra,yl1,rival
0,25,1,1991,1,1,1.0,1.0,2,3.0,16.0,...,8223.587891,0.0,12.0,NaN,NaN,14460.792969,45.0,6.0,25306.388672,2.115668
1,25,1,1993,1,1,1.0,1.0,2,3.0,16.0,...,3342.373779,0.0,12.0,NaN,7.0,16526.621094,39.0,6.0,27888.671875,1.908828
2,25,1,1995,1,1,1.0,1.0,2,3.0,16.0,...,4761.857422,0.0,12.0,NaN,7.0,23240.560547,45.0,6.0,36151.984375,1.700596


## 2. Rename Variables (Italian → English)

In [3]:
rename_map = {
    "sesso":   "sex",
    "anasc":   "yearbirth",
    "staciv":  "marsta",
    "studio":  "educ",
    "qualp7n": "empsta",
    "settp7":  "worksec",
    "nonoc":   "notempsta",
    "eta":     "age",
    "anno":    "year",
    "cfdic":   "hhead",
}

# Only rename columns that actually exist
actual_rename = {k: v for k, v in rename_map.items() if k in df.columns}
df = df.rename(columns=actual_rename)
print("Renamed:", actual_rename)

Renamed: {'sesso': 'sex', 'anasc': 'yearbirth', 'staciv': 'marsta', 'studio': 'educ', 'qualp7n': 'empsta', 'settp7': 'worksec', 'nonoc': 'notempsta', 'eta': 'age', 'anno': 'year', 'cfdic': 'hhead'}


## 3. Add Value Labels
Pandas uses categoricals for labeled variables (equivalent to Stata's `label define / label values`).

In [4]:
# sex: 1=Men, 2=Women
if "sex" in df.columns:
    sex_labels = {1: "Men", 2: "Women"}
    df["sex_label"] = df["sex"].map(sex_labels)
    print("sex distribution:")
    print(df["sex_label"].value_counts().to_string(), "\n")

# marital status: 1=Married 2=Single 3=Divorced/separated 4=Widowed
if "marsta" in df.columns:
    marsta_labels = {1: "Married", 2: "Single (never married)",
                     3: "Divorced/separated", 4: "Widowed"}
    df["marsta_label"] = df["marsta"].map(marsta_labels)
    print("marsta distribution:")
    print(df["marsta_label"].value_counts().to_string(), "\n")

# employment status
if "empsta" in df.columns:
    empsta_labels = {
        1: "Blue-collar worker", 2: "Office worker/teacher",
        3: "Cadre or manager",   4: "Sole proprietor/arts/professions",
        5: "Other self-employed",6: "Pensioner",
        7: "Other not-employed"
    }
    df["empsta_label"] = df["empsta"].map(empsta_labels)
    print("empsta distribution:")
    print(df["empsta_label"].value_counts().to_string())

sex distribution:
sex_label
Women    159864
Men      152039 

marsta distribution:
marsta_label
Married                   159258
Single (never married)    118271
Widowed                    24737
Divorced/separated          9637 

empsta distribution:
empsta_label
Other not-employed                  202163
Blue-collar worker                   38820
Office worker/teacher                38090
Pensioner                            18526
Other self-employed                   6175
Cadre or manager                      5487
Sole proprietor/arts/professions      2642


## 4. Deflate Monetary Variables to 2020 Prices
Multiply each monetary variable by `rival` (revaluation coefficient to 2020).

In [5]:
DEFLATE_VARS = ["w", "c", "consal", "condiv", "cd", "cd1", "cd2",
                "ar", "ar1", "ar2", "ar3", "ylm", "s1", "y1"]

deflated = []
for var in DEFLATE_VARS:
    if var in df.columns and "rival" in df.columns:
        df[var] = df[var] * df["rival"]
        deflated.append(var)

print(f"Deflated to 2020 prices: {deflated}")

Deflated to 2020 prices: ['w', 'c', 'cd', 'cd1', 'cd2', 'ar', 'ar1', 'ar2', 'ar3', 'ylm', 's1', 'y1']


## 5. Dummy Variables

In [6]:
# married: 1 if married, 0 otherwise
if "marsta" in df.columns:
    df["married"] = np.where(df["marsta"] == 1, 1,
                    np.where(df["marsta"].isna(), np.nan, 0))

# employed: 1 if empsta <= 5 (active worker), 0 otherwise
if "empsta" in df.columns:
    df["employed"] = np.where(df["empsta"] <= 5, 1,
                     np.where(df["empsta"].isna(), np.nan, 0))

# female: 1 if woman
if "sex" in df.columns:
    df["female"] = np.where(df["sex"] == 2, 1,
                   np.where(df["sex"].isna(), np.nan, 0))

# graduate: educ > 4 (tertiary or post-grad)
if "educ" in df.columns:
    df["graduate"] = np.where(df["educ"] > 4, 1,
                     np.where(df["educ"].isna(), np.nan, 0))

# retired, unemployed dummies (from notempsta)
if "notempsta" in df.columns:
    df["retired"]  = (df["notempsta"] == 5).astype(float)
    df["unempl"]   = (df["notempsta"] == 6).astype(float)
    df.loc[df["notempsta"].isna(), ["retired", "unempl"]] = np.nan

# Geographic dummies (area3: 1=North, 2=Centre, 3=South)
if "area3" in df.columns:
    df["centreitaly"] = np.where(df["area3"] == 2, 1,
                        np.where(df["area3"].isna(), np.nan, 0))
    df["southitaly"]  = np.where(df["area3"] == 3, 1,
                        np.where(df["area3"].isna(), np.nan, 0))

print("Dummy variables created: married, employed, female, graduate, retired, unempl, centreitaly, southitaly")

Dummy variables created: married, employed, female, graduate, retired, unempl, centreitaly, southitaly


## 6. Age Squared

In [7]:
if "age" in df.columns:
    df["age2"] = df["age"] ** 2
    print("age2 created")

age2 created


## 7. Birth Cohorts (by decade)

In [8]:
if "yearbirth" in df.columns:
    # cohort: 7 groups from 1920s to 1980s
    cohort_bins   = [1919, 1929, 1939, 1949, 1959, 1969, 1979, 1989]
    cohort_labels = [1, 2, 3, 4, 5, 6, 7]
    cohort_str    = ["1920-1929","1930-1939","1940-1949","1950-1959",
                     "1960-1969","1970-1979","1980-1989"]
    df["cohort"]  = pd.cut(df["yearbirth"], bins=cohort_bins,
                           labels=cohort_labels, right=True)
    df["cohort2"] = pd.cut(df["yearbirth"], bins=cohort_bins,
                           labels=cohort_str, right=True)

    print("Cohort distribution:")
    print(df["cohort2"].value_counts().sort_index().to_string())

Cohort distribution:
cohort2
1920-1929    19619
1930-1939    33996
1940-1949    43228
1950-1959    46484
1960-1969    47650
1970-1979    40246
1980-1989    31715


## 8. Age Groups (5-year and 10-year bands)

In [9]:
if "age" in df.columns:

    # ── 5-year bands (agegr5) ───────────────────────────────────────────────
    agegr5_bins   = [24, 29, 34, 39, 44, 49, 54, 59, 64, 69, 74, 79, 84]
    agegr5_labels = list(range(1, 13))
    agegr5_str    = ["25-29","30-34","35-39","40-44","45-49","50-54",
                     "55-59","60-64","65-69","70-74","75-79","80-84"]
    df["agegr5"] = pd.cut(df["age"], bins=agegr5_bins,
                          labels=agegr5_labels, right=True)

    # ── 10-year bands (agegr10) ─────────────────────────────────────────────
    agegr10_bins  = [-np.inf, 24, 34, 44, 54, 64, 74, 84, np.inf]
    agegr10_labels = ["<25","25-34","35-44","45-54","55-64","65-74","75-84","85+"]
    df["agegr10"] = pd.cut(df["age"], bins=agegr10_bins,
                           labels=agegr10_labels, right=True)

    print("10-year age group distribution:")
    print(df["agegr10"].value_counts().sort_index().to_string())

10-year age group distribution:
agegr10
<25      77611
25-34    36293
35-44    39372
45-54    45960
55-64    44723
65-74    38261
75-84    23021
85+       6662


## 9. Number of Children by Age Group
`par == 3` identifies children in the household. We count them by age bracket per household-year.

In [10]:
if "par" in df.columns and "age" in df.columns:

    # Flag each child in each age bracket
    df["child3"]     = np.where((df["par"] == 3) & (df["age"] <= 3),  1, 0)
    df["child4_10"]  = np.where((df["par"] == 3) & (df["age"] >= 4)  & (df["age"] <= 10), 1, 0)
    df["child11_19"] = np.where((df["par"] == 3) & (df["age"] >= 11) & (df["age"] <= 19), 1, 0)

    # Set to NaN where par or age is missing
    for c in ["child3", "child4_10", "child11_19"]:
        df.loc[df["par"].isna() | df["age"].isna(), c] = np.nan

    # Aggregate per household-year (bysort nquest year)
    key = ["nquest", "year"] if "year" in df.columns else ["nquest", "anno"]
    for c in ["child3", "child4_10", "child11_19"]:
        df[f"n{c}"] = df.groupby(key)[c].transform("sum")

    # Binary 'children' indicator
    df["children"] = ((df["child3"] == 1) |
                      (df["child4_10"] == 1) |
                      (df["child11_19"] == 1)).astype(int)

    print("Children variables created: nchild3, nchild4_10, nchild11_19, children")
    print(df["children"].value_counts().to_string())

Children variables created: nchild3, nchild4_10, nchild11_19, children
children
0    256866
1     55037


## 10. Final Sort & Save

In [11]:
year_col = "year" if "year" in df.columns else "anno"
df = df.sort_values(["nquest", "nord", year_col]).reset_index(drop=True)

out_pkl = os.path.join(DataOut, "dataready_SHIW2022.pkl")
out_csv = os.path.join(DataOut, "dataready_SHIW2022.csv")

df.to_pickle(out_pkl)
df.to_csv(out_csv, index=False)

print(f"Saved to: {out_pkl}")
print(f"Final dataset: {df.shape[0]:,} rows × {df.shape[1]} columns")

Saved to: C:/Users/Utente/OneDrive/Immagini/Documenti/Economic_Risk_work\DataOut\dataready_SHIW2022.pkl
Final dataset: 311,903 rows × 71 columns


## 11. Summary Statistics (Quick Check)

In [12]:
key_vars = ["age", "female", "married", "graduate", "employed",
            "y1", "c", "s1", "w", "ylm"]
available = [v for v in key_vars if v in df.columns]

print("Summary statistics (key variables):")
df[available].describe().round(2)

Summary statistics (key variables):


,age,female,married,graduate,employed,y1,c,s1,w,ylm
count,311903.00,311903.00,311903.00,311903.00,311903.00,311903.00,311903.00,311903.00,3.119030e+05,87778.00
mean,43.86,0.51,0.51,0.09,0.29,45202.06,32471.44,12730.62,3.429605e+05,20031.60
std,22.87,0.50,0.50,0.29,0.45,47131.11,21226.78,37467.95,1.746686e+06,13654.02
min,0.00,0.00,0.00,0.00,0.00,-199681.47,-58019.79,-462221.69,-1.734362e+06,0.00
25%,25.00,0.00,0.00,0.00,0.00,24038.33,19700.00,1022.43,4.777898e+04,14140.41
50%,45.00,1.00,1.00,0.00,0.00,37189.29,27534.81,7309.57,1.835693e+05,19146.60
75%,62.00,1.00,1.00,0.00,1.00,55300.00,39366.51,18274.61,3.599280e+05,23980.90
max,104.00,1.00,1.00,1.00,1.00,6197000.00,611538.12,5871280.00,3.407900e+08,1200000.00


In [13]:
print("\nSample size by year:")
print(df[year_col].value_counts().sort_index().to_string())
print("\nNotebook 2 — Data Preparation complete ✓")


Sample size by year:
year
1991    24930
1993    24050
1995    23955
1998    20937
2000    22382
2002    21195
2004    20622
2006    19621
2008    19961
2010    19885
2012    20076
2014    19400
2016    16508
2020    15248
2022    23133

Notebook 2 — Data Preparation complete ✓
